# YOLO Foundations Part 2: Real-World Solutions

**Format:** Station-based Exercises + Group Capstone

In this lab, we'll explore **Ultralytics Solutions** — high-level wrappers that transform raw detections into actionable business insights.

## Learning Objectives
- Apply region-based counting with custom polygons
- Measure speed and distance between tracked objects
- Generate analytics charts and heatmaps from video data
- Implement privacy-compliant object blurring
- Combine multiple solutions into a complete pipeline

---

## Video Sources

We'll use sample videos from local files and GitHub:

| Source | File | Use Case |
|--------|------|----------|
| **Local** | `Pull_ups.mp4` | Workout monitoring |
| **Local** | `parking_slots.mp4` | Parking management |
| **Local** | `Cars.mp4` | Sample traffic video |

---

## 1. Environment Setup

In [ ]:
# Install Ultralytics and import libraries
import sys
%pip install -q ultralytics opencv-python pandas matplotlib requests
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import os
import requests
from ultralytics import YOLO, solutions

# Download videos from GitHub if not exists locally
videos = ["Pull_ups.mp4", "parking_slots.mp4", "Cars.mp4"]
# Updated loop logic
for video_file in videos:
    if not os.path.exists(video_file):
        # Using the direct download link structure
        url = f"https://github.com/NaifMersal/cv-for-developers-ultralytics/raw/main/labs/{video_file}"
        
        # Adding stream=True is better for large video files
        r = requests.get(url, stream=True)
        
        if r.status_code == 200:
            with open(video_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"✓ Downloaded: {video_file}")
        else:
            print(f"✗ Failed to download {video_file}. Status code: {r.status_code}")

---

# Station 1: Region & Boundary Analytics (40 min)

## 📍 Exercise 1.1: Custom Counting Region (15 min)

**Task:** Count vehicles crossing a specific lane using a custom polygon region. Compare results between a line and a polygon.

**Video:** Urban traffic intersection

In [ ]:
# 🚗 Exercise 1.1: Object Counting with Custom Region

video_path = "Cars.mp4"

# Download if not exists
if not os.path.exists(video_path):
    url = f"https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/labs/{video_path}"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)

# Load model and open video
model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
print(f"Video: {w}x{h} @ {fps} fps")

# Define counting region - TRY modifying these coordinates!
# For a LINE: two points like [(w//2, h//2), (w//2+100, h//2)]
# For a POLYGON: 3+ points like [(x1,y1), (x2,y2), (x3,y3), (x4,y4)]

# TODO: Create a polygon that captures only ONE lane of traffic
line_region = [(w//2 - 200, h//2), (w//2 + 200, h//2)]  # 🔧 FIX THIS

counter = solutions.ObjectCounter(
    show=True,
    region=line_region,
    model=model,
    classes=[2, 3, 5, 7]  # car, motorcycle, bus, truck
)

# Process video (limit frames for demo)
frame_count = 0
max_frames = 150

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    results = counter(im0)
    frame_count += 1

cap.release()

print(f"\n📊 Results from {frame_count} frames:")
print(f"   Total IN:  {counter.in_count}")
print(f"   Total OUT: {counter.out_count}")

### 🏆 Challenge (Optional)
Try different region shapes:
- A horizontal line across half the frame
- A rectangle covering only the left lane
- Which gives more accurate counts?

---

## 🅿️ Exercise 1.2: Parking Lot Management (15 min)

**Task:** Define parking spots and track occupancy over time. Log results to CSV.

**Video:** Drone footage of mall parking lot

In [ ]:
# 🅿️ Exercise 1.2: Parking Management

video_path = "parking_slots.mp4"

# Download if not exists
if not os.path.exists(video_path):
    url = f"https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/labs/{video_path}"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Video: {w}x{h}")

# Define 4 parking spots as polygons
# Each spot = list of 4 corner coordinates
parking_spots = [
    [(50, 200), (200, 200), (200, 350), (50, 350)],   # Spot 1
    [(220, 200), (370, 200), (370, 350), (220, 350)], # Spot 2
    [(390, 200), (540, 200), (540, 350), (390, 350)], # Spot 3
    [(560, 200), (710, 200), (710, 350), (560, 350)], # Spot 4
]

manager = solutions.ParkingManagement(
    show=True,
    region=parking_spots,
    model=model,
    classes=[2, 3, 5, 7]  # vehicles
)

# Track occupancy over time
occupancy_log = []
frame_count = 0
max_frames = 100

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    results = manager(im0)
    
    # Log occupancy
    occupied = sum(1 for spot in manager.parking_slots if spot['occupied'])
    occupancy_log.append({'frame': frame_count, 'occupied': occupied, 'total': len(parking_spots)})
    
    frame_count += 1

cap.release()

# Save to CSV
df = pd.DataFrame(occupancy_log)
df.to_csv('parking_occupancy.csv', index=False)
print(f"\n📊 Parking Summary:")
print(f"   Frames processed: {frame_count}")
print(f"   Max occupancy:  {df['occupied'].max()}/{len(parking_spots)}")
print(f"   Saved to: parking_occupancy.csv")

### 🏆 Challenge
Add a real-time alert when occupancy exceeds 75%

---

## 📋 Exercise 1.3: Queue Alert System (10 min)

**Task:** Monitor a region and trigger alert when queue exceeds threshold.

In [ ]:
# 📋 Exercise 1.3: Queue Alert System

# Use the same traffic video - define a zone as "queue area"
video_path = "Cars.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)

# Define queue region (polygon in center of frame)
queue_region = [(400, 300), (900, 300), (900, 600), (400, 600)]

# Use ObjectCounter in region mode
queue_counter = solutions.ObjectCounter(
    show=True,
    region=queue_region,
    model=model,
    classes=[2, 3, 5, 7]
)

ALERT_THRESHOLD = 5
alert_triggered = False

frame_count = 0
max_frames = 100

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    results = queue_counter(im0)
    
    current_count = queue_counter.in_count + queue_counter.out_count
    
    if current_count > ALERT_THRESHOLD and not alert_triggered:
        print(f"🚨 ALERT: Queue exceeded {ALERT_THRESHOLD} at frame {frame_count}!")
        alert_triggered = True
    
    frame_count += 1

cap.release()
print(f"\n✓ Processed {frame_count} frames")

---

# Station 2: Kinematic & Spatial Measurement (35 min)

## 🚀 Exercise 2.1: Speed Estimation (20 min)

**Task:** Estimate vehicle speeds by calibrating `meter_per_pixel`. Identify the fastest vehicle.

**Hint:** Start with `meter_per_pixel=0.05` and adjust based on visual reference.

In [ ]:
# 🚀 Exercise 2.1: Speed Estimation

video_path = "Cars.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

fps = int(cap.get(cv2.CAP_PROP_FPS))

# TODO: Calibrate this value! 
# Estimate: if road is ~10 meters wide, measure pixels and divide
METER_PER_PIXEL = 0.05  # 🔧 ADJUST THIS

speed_est = solutions.SpeedEstimator(
    show=True,
    model=model,
    fps=fps,
    meter_per_pixel=METER_PER_PIXEL,
    classes=[2, 3, 5, 7]
)

# Track speed readings
speed_readings = []
frame_count = 0
max_frames = 100

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    results = speed_est(im0)
    
    # Access speed data from the results
    if hasattr(results, 'speed') and results.speed is not None:
        for track_id, speed in results.speed.items():
            if speed > 0:
                speed_readings.append({'frame': frame_count, 'track_id': track_id, 'speed_kmh': speed})
    
    frame_count += 1

cap.release()

# Find fastest vehicle
if speed_readings:
    df = pd.DataFrame(speed_readings)
    fastest = df.loc[df['speed_kmh'].idxmax()]
    print(f"\n🏎️ Fastest vehicle:")
    print(f"   Track ID: {int(fastest['track_id'])}")
    print(f"   Speed: {fastest['speed_kmh']:.1f} km/h")
    
    # Summary stats
    print(f"\n📊 Speed Summary:")
    print(f"   Avg speed: {df['speed_kmh'].mean():.1f} km/h")
    print(f"   Max speed: {df['speed_kmh'].max():.1f} km/h")
else:
    print("⚠️ No speed data captured")

### 🏆 Challenge
Adjust `meter_per_pixel` to get more realistic speeds. What value works best?

---

## 📏 Exercise 2.2: Distance Calculation (15 min)

**Task:** Calculate distance between two tracked vehicles. Trigger warning if too close.

In [ ]:
# 📏 Exercise 2.2: Distance Calculation

video_path = "Cars.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

dist_calc = solutions.DistanceCalculation(
    show=True,
    model=model,
    classes=[2, 3, 5, 7]
)

DISTANCE_THRESHOLD_METERS = 2.0
warnings = []
frame_count = 0
max_frames = 80

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    results = dist_calc(im0)
    
    # Check for close distances
    if hasattr(results, 'data') and results.data is not None:
        distances = results.data
        if len(distances) >= 2:
            # Get pairwise distances
            for i in range(len(distances)-1):
                d = distances[i].dist
                if d > 0 and d < DISTANCE_THRESHOLD_METERS * 100:  # Convert to cm
                    warnings.append({'frame': frame_count, 'distance_cm': d})
                    print(f"⚠️ Frame {frame_count}: Vehicles too close! ({d:.1f} cm)")
    
    frame_count += 1

cap.release()
print(f"\n✓ Processed {frame_count} frames")
print(f"⚠️ Total close-proximity events: {len(warnings)}")

---

# Station 3: Privacy, Analytics & Visualization (35 min)

## 🔒 Exercise 3.1: Object Blurring (10 min)

**Task:** Blur detected persons for privacy compliance.

**Challenge:** Try to blur only faces using pose keypoints (advanced).

In [ ]:
# 🔒 Exercise 3.1: Object Blurring

# Using a pedestrian video
video_path = "parking_slots.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

blurrer = solutions.ObjectBlurrer(
    show=True,
    model=model,
    blur_ratio=0.5,  # 0-1, higher = more blur
    classes=[0]  # person only
)

frame_count = 0
max_frames = 50

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    results = blurrer(im0)
    frame_count += 1

cap.release()
print(f"✓ Blurred {frame_count} frames (persons only)")

### 🏆 Advanced Challenge: Face-Only Blur
Use YOLO Pose (`yolov8n-pose.pt`) and blur only keypoints 0-4 (face)

---

## 📈 Exercise 3.2: Analytics Dashboard (15 min)

**Task:** Generate line/bar/pie charts showing object class distribution over time.

In [ ]:
# 📈 Exercise 3.2: Analytics Dashboard

video_path = "Cars.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

# Try different analytics types: line, bar, area, pie
analytics = solutions.Analytics(
    show=True,
    analytics_type="line",  # change to "bar", "pie", "area"
    model=model,
    classes=[2, 3, 5, 7]
)

frame_count = 0
max_frames = 100

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    results = analytics(im0, frame_count=frame_count)
    frame_count += 1

cap.release()

# Save analytics data
if hasattr(analytics, 'results'):
    print(f"\n📊 Analytics generated for {frame_count} frames")
    print(f"   Type: {analytics.analytics_type}")
    
    # Export data if available
    try:
        analytics.save_json("analytics_data.json")
        print("   Saved: analytics_data.json")
    except:
        pass

### 🏆 Challenge
Create all 4 chart types (line, bar, pie, area) and identify peak hours

---

## 🔥 Exercise 3.3: Heatmap Visualization (10 min)

**Task:** Generate a heatmap showing movement density. Compare colormaps.

In [ ]:
# 🔥 Exercise 3.3: Heatmap Visualization

video_path = "parking_slots.mp4"

model = YOLO("weights/yolo26n.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

heatmap = solutions.Heatmap(
    show=True,
    model=model,
    colormap=cv2.COLORMAP_JET,  # Try: JET, VIRIDIS, HOT, TURBO
    classes=[0]  # persons
)

frame_count = 0
max_frames = 100

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    results = heatmap.generate_heatmap(imgsz=(h, w), boxes=model.predict(im0, verbose=False)[0].boxes)
    frame_count += 1

cap.release()
print(f"✓ Generated heatmap from {frame_count} frames")

---

# Station 4: Specialized Heads (30 min)

## 🏋️ Exercise 4.1: Workout Monitoring (15 min)

**Task:** Count pushups using YOLO Pose. Modify for squats or jumping jacks.

**Video:** Person doing pull-ups

In [ ]:
# 🏋️ Exercise 4.1: Workout Monitoring

video_path = "Pull_ups.mp4"

# Download if not exists
if not os.path.exists(video_path):
    url = f"https://raw.githubusercontent.com/NaifMersal/cv-for-developers-ultralytics/main/labs/{video_path}"
    r = requests.get(url)
    with open(video_path, "wb") as f:
        f.write(r.content)

model = YOLO("yolov8n-pose.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

gym = solutions.AIGym(
    show=True,
    kpts=[6, 8, 10],  # Keypoints for pushup: shoulder-elbow-wrist (left arm)
    model=model
)

frame_count = 0
max_frames = 100

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    results = gym(im0)
    frame_count += 1

cap.release()

print(f"\n💪 Workout Summary:")
print(f"   Frames processed: {frame_count}")
if hasattr(gym, 'count'):
    print(f"   Rep count: {gym.count}")

### 🏆 Challenge
Modify the keypoints for:
- **Squats:** kpts=[11, 13, 15] (hip-knee-ankle)
- **Jumping jacks:** kpts=[5, 7, 9] + [6, 8, 10] (both arms)

---

## 🎭 Exercise 4.2: Instance Segmentation Tracking (15 min)

**Task:** Track objects with pixel-perfect masks. Compare to bounding box tracking.

In [ ]:
# 🎭 Exercise 4.2: Instance Segmentation Tracking

video_path = "Cars.mp4"

# Use YOLO Segmentation model
model = YOLO("yolo26n-seg.pt")
cap = cv2.VideoCapture(video_path)
assert cap.isOpened(), "Error loading video"

w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

frame_count = 0
max_frames = 50

# Track using segmentation model
results = model.track(source=video_path, stream=True, classes=[2, 3, 5, 7], persist=True)

tracked_objects = set()

for result in results:
    if result.boxes.id is not None:
        for track_id in result.boxes.id.cpu().numpy():
            tracked_objects.add(int(track_id))
    frame_count += 1
    if frame_count >= max_frames:
        break

print(f"✓ Tracked {len(tracked_objects)} unique vehicles with instance masks")

---

# Capstone Project: Smart Retail Analytics System (60 min)

## 🎯 Group Challenge (3-4 students)

Build a complete retail analytics system that:

1. **Counts customers** entering the store (ObjectCounter)
2. **Generates heatmap** of popular aisles (Heatmap)
3. **Blurs faces** for privacy (ObjectBlurrer)
4. **Shows peak hours** with analytics chart (Analytics)
5. **Alerts** if checkout queue > 3 people (Queue alert)

### Requirements
- Use **at least 3 different solutions**
- Process video end-to-end
- Export results to CSV/JSON

### Stretch Goals
- Add speed estimation for customer walking patterns
- Create a simple dashboard visualization
- Combine multiple videos

### Deliverables
- Working notebook with all solutions integrated
- `retail_summary.csv` with counts, peak times, occupancy
- 2-minute demo per group

In [ ]:
# 🎯 CAPSTONE: Smart Retail Analytics System

# Use parking video as proxy for retail store
video_path = "parking_slots.mp4"

model_det = YOLO("weights/yolo26n.pt")
model_pose = YOLO("yolov8n-pose.pt")

cap = cv2.VideoCapture(video_path)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Initialize all solutions
counter = solutions.ObjectCounter(show=False, region=[(w//2, h//2-100), (w//2, h//2+100)], model=model_det, classes=[0])
blurrer = solutions.ObjectBlurrer(show=False, blur_ratio=0.6, model=model_det, classes=[0])
heatmap = solutions.Heatmap(show=False, model=model_det, classes=[0])
analytics = solutions.Analytics(show=False, analytics_type="line", model=model_det, classes=[0])

# Data collection
counts = []
heatmap_frames = []

frame_count = 0
max_frames = 150

while cap.isOpened() and frame_count < max_frames:
    success, im0 = cap.read()
    if not success:
        break
    
    # 1. Count customers
counter.start_counting(frame=im0, boxes=model_det.predict(im0, verbose=False)[0].boxes)
    
    # 2. Blur faces (use pose keypoints for face detection)
    blur_result = blurrer(im0)
    
    # 3. Generate heatmap
    det_result = model_det.predict(im0, verbose=False)[0]
    heatmap.generate_heatmap(imgsz=(h, w), boxes=det_result.boxes)
    
    # 4. Analytics
    analytics(im0, frame_count=frame_count)
    
    # Log data
    counts.append({'frame': frame_count, 'in': counter.in_count, 'out': counter.out_count})
    
    frame_count += 1

cap.release()

# Export results
df = pd.DataFrame(counts)
df.to_csv('retail_summary.csv', index=False)

print("=" * 50)
print("🎯 CAPSTONE COMPLETE!")
print("=" * 50)
print(f"\n📊 Results:")
print(f"   Total customers in:  {counter.in_count}")
print(f"   Total customers out: {counter.out_count}")
print(f"   Frames processed:    {frame_count}")
print(f"\n📁 Exports:")
print(f"   - retail_summary.csv")
print(f"   - heatmap output")
print(f"   - analytics data")

---

## ✅ Wrap-up Questions

1. Which solution was most challenging to configure?
2. How would you adapt these solutions for a real deployment?
3. What other business problems could these solutions solve?

---

## 📚 References

- [Object Counting Docs](https://docs.ultralytics.com/guides/object-counting/)
- [Heatmaps Docs](https://docs.ultralytics.com/guides/heatmaps/)
- [Speed Estimation Docs](https://docs.ultralytics.com/guides/speed-estimation/)
- [Analytics Docs](https://docs.ultralytics.com/guides/analytics/)
- [Parking Management Docs](https://docs.ultralytics.com/guides/parking-management/)
- [Workouts Monitoring Docs](https://docs.ultralytics.com/guides/workouts-monitoring/)

---

**End of Lab**